# 04 — Cross-asset TSMOM backtest

**Setup:** rank 17 ETFs by 12-month trailing return. Long positive, short negative
(or cash if `allow_short=False`). Vol-target each leg to 10%. Rebalance monthly.

**Bias controls:** signal uses data only up to month-end t; trades execute at next
month-start prices.

In [ ]:
import sys, os, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath(".."))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
from src.data.yfinance_client import YFinanceClient
from src.strategies.tsmom import TSMOMStrategy, TSMOMConfig
from src.backtest.engine import run_backtest, EngineConfig
from src.reports.tearsheet import tearsheet

yf = YFinanceClient()
universe = ['SPY','QQQ','IWM','TLT','IEF','GLD','SLV','USO','VNQ','XLE','XLK']
data = {s: yf.get_daily_bars(s, start='2008-01-01')['adj_close']
         for s in universe}
prices = pd.DataFrame(data).ffill().dropna()
print('Panel:', prices.shape)

### Run

In [ ]:
strat = TSMOMStrategy(TSMOMConfig(universe=list(prices.columns),
                                   per_asset_vol=0.08,
                                   max_gross_leverage=1.0, allow_short=True))
res = run_backtest(strat, prices,
                   EngineConfig(starting_cash=100_000, rebalance_freq='M',
                                 benchmark='SPY', strategy_name='tsmom',
                                 run_preflight=False))
res['metrics']

In [ ]:
fig, m = tearsheet(res, 'TSMOM cross-asset')
plt.show()